#Modelo para predeicion ganador del Mundial 2026

**Reasoning**:
I will modify the `pd.read_csv` call in the existing code cell to include `encoding='latin1'` to resolve the `UnicodeDecodeError`.



In [ ]:
import pandas as pd
import numpy as np
from google.colab import drive
drive.mount('/content/drive')
file_path = "/content/drive/MyDrive/FIFA World Cup 1930-2022 All Match Dataset.csv"
df_Mundial = pd.read_csv(file_path, encoding='latin1')
df_Mundial.head()

Mounted at /content/drive


,Key Id,Tournament Id,tournament Name,Match Id,Match Name,Stage Name,Group Name,Group Stage,Knockout Stage,Replayed,...,Away Team Score Margin,Extra Time,Penalty Shootout,Score Penalties,Home Team Score Penalties,Away Team Score Penalties,Result,Home Team Win,Away Team Win,Draw
0,1,WC-1930,1930 FIFA World Cup,M-1930-01,France v Mexico,group stage,Group 1,1,0,0,...,-3,0,0,0-0,0,0,home team win,1,0,0
1,2,WC-1930,1930 FIFA World Cup,M-1930-02,United States v Belgium,group stage,Group 4,1,0,0,...,-3,0,0,0-0,0,0,home team win,1,0,0
2,3,WC-1930,1930 FIFA World Cup,M-1930-03,Yugoslavia v Brazil,group stage,Group 2,1,0,0,...,-1,0,0,0-0,0,0,home team win,1,0,0
3,4,WC-1930,1930 FIFA World Cup,M-1930-04,Romania v Peru,group stage,Group 3,1,0,0,...,-2,0,0,0-0,0,0,home team win,1,0,0
4,5,WC-1930,1930 FIFA World Cup,M-1930-05,Argentina v France,group stage,Group 1,1,0,0,...,-1,0,0,0-0,0,0,home team win,1,0,0


In [ ]:

#Numero de registros
Registros = df_Mundial.shape
print(f"El numero de registros es: {Registros}")

El numero de registros es: (964, 37)


In [ ]:
#Verificacion valores nulos
df_Mundial.isnull().sum()

,0
Key Id,0
Tournament Id,0
tournament Name,0
Match Id,0
Match Name,0
Stage Name,0
Group Name,0
Group Stage,0
Knockout Stage,0
Replayed,0


In [ ]:
#Informacion base de datos
df_Mundial.info ()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 964 entries, 0 to 963
Data columns (total 37 columns):
 #   Column                     Non-Null Count  Dtype 
---  ------                     --------------  ----- 
 0   Key Id                     964 non-null    int64 
 1   Tournament Id              964 non-null    object
 2   tournament Name            964 non-null    object
 3   Match Id                   964 non-null    object
 4   Match Name                 964 non-null    object
 5   Stage Name                 964 non-null    object
 6   Group Name                 964 non-null    object
 7   Group Stage                964 non-null    int64 
 8   Knockout Stage             964 non-null    int64 
 9   Replayed                   964 non-null    int64 
 10  Replay                     964 non-null    int64 
 11  Match Date                 964 non-null    object
 12  Match Time                 964 non-null    object
 13  Stadium Id                 964 non-null    object
 14  Stadium Na

In [ ]:
#Creacion de Columnda Resultados
df_Mundial["resultado"] = df_Mundial.apply(
    lambda row: 1 if row["Home Team Score"] > row["Away Team Score"]
    else (-1 if row["Home Team Score"] < row["Away Team Score"] else 0),
    axis=1
)
df_Mundial[["Home Team Score", "Away Team Score", "resultado"]].head(10)

,Home Team Score,Away Team Score,resultado
0,4,1,1
1,3,0,1
2,2,1,1
3,3,1,1
4,1,0,1
5,3,0,1
6,4,0,1
7,3,0,1
8,1,0,1
9,1,0,1


In [ ]:
#Ver distribuciones de clases
df_Mundial["resultado"].value_counts()
df_Mundial["resultado"].value_counts(normalize=True)

,proportion
resultado,
1,0.543568
-1,0.234440
0,0.221992


In [ ]:
#Pasar fecha a datetime
df_Mundial["Match Date"] = pd.to_datetime(df_Mundial["Match Date"])

In [ ]:
#Reet en el indice para evitar problemas
df_Mundial = df_Mundial.reset_index(drop=True)

In [ ]:
#Comprobar si quedo ordenado
df_Mundial[["Match Date"]].head() #1930


,Match Date
0,1930-07-13
1,1930-07-13
2,1930-07-14
3,1930-07-14
4,1930-07-15


In [ ]:
#Comprobar si quedo ordenado
df_Mundial[["Match Date"]].tail() #2022

,Match Date
959,2022-12-10
960,2022-12-13
961,2022-12-14
962,2022-12-17
963,2022-12-18


In [ ]:
#Ordenamos data set
df_Mundial = df_Mundial.sort_values("Match Date").reset_index(drop=True)

In [ ]:
#Creamos como estara la estructura para cumular esttaidistca
team_stats = {
  "matches": 0,
  "goals_scored": 0,
  "goals_conceded": 0,
  "wins": 0
}

In [ ]:
# Creamos donde se gaurdara todo
home_avg_goals = []
away_avg_goals = []
home_win_rate = []
away_win_rate = []

In [ ]:
# Sacamos estadisticas por cada partido
team_stats = {}

home_avg_goals = []
away_avg_goals = []

home_avg_conceded = []
away_avg_conceded = []

home_win_rate = []
away_win_rate = []

for idx, row in df_Mundial.iterrows():

    home = row["Home Team Name"]
    away = row["Away Team Name"]

    # Inicializar equipos si no existen
    if home not in team_stats:
        team_stats[home] = {
            "matches": 0,
            "goals_scored": 0,
            "goals_conceded": 0,
            "wins": 0
        }

    if away not in team_stats:
        team_stats[away] = {
            "matches": 0,
            "goals_scored": 0,
            "goals_conceded": 0,
            "wins": 0
        }

    # ---- CALCULAR ESTADÍSTICAS PREVIAS (ANTES DEL PARTIDO) ----

    home_matches = team_stats[home]["matches"]
    away_matches = team_stats[away]["matches"]

    # Promedio goles anotados
    home_avg = team_stats[home]["goals_scored"] / home_matches if home_matches > 0 else 0
    away_avg = team_stats[away]["goals_scored"] / away_matches if away_matches > 0 else 0

    # Promedio goles concedidos
    home_conceded_avg = team_stats[home]["goals_conceded"] / home_matches if home_matches > 0 else 0
    away_conceded_avg = team_stats[away]["goals_conceded"] / away_matches if away_matches > 0 else 0

    # Win rate
    home_wr = team_stats[home]["wins"] / home_matches if home_matches > 0 else 0
    away_wr = team_stats[away]["wins"] / away_matches if away_matches > 0 else 0

    # Guardar features
    home_avg_goals.append(home_avg)
    away_avg_goals.append(away_avg)

    home_avg_conceded.append(home_conceded_avg)
    away_avg_conceded.append(away_conceded_avg)

    home_win_rate.append(home_wr)
    away_win_rate.append(away_wr)

    # ---- ACTUALIZAR ESTADÍSTICAS DESPUÉS DEL PARTIDO ----

    team_stats[home]["matches"] += 1
    team_stats[away]["matches"] += 1

    team_stats[home]["goals_scored"] += row["Home Team Score"]
    team_stats[home]["goals_conceded"] += row["Away Team Score"]

    team_stats[away]["goals_scored"] += row["Away Team Score"]
    team_stats[away]["goals_conceded"] += row["Home Team Score"]

    if row["resultado"] == 1:
        team_stats[home]["wins"] += 1
    elif row["resultado"] == -1:
        team_stats[away]["wins"] += 1

In [ ]:
# se agregan columnas al data set
df_Mundial["home_avg_goals"] = home_avg_goals
df_Mundial["away_avg_goals"] = away_avg_goals

df_Mundial["home_avg_conceded"] = home_avg_conceded
df_Mundial["away_avg_conceded"] = away_avg_conceded

df_Mundial["home_win_rate"] = home_win_rate
df_Mundial["away_win_rate"] = away_win_rate

In [ ]:
# este paso se crean las diferencias
df_Mundial["diff_avg_goals"] = df_Mundial["home_avg_goals"] - df_Mundial["away_avg_goals"]
df_Mundial["diff_win_rate"] = df_Mundial["home_win_rate"] - df_Mundial["away_win_rate"]
df_Mundial["diff_avg_conceded"] = df_Mundial["home_avg_conceded"] - df_Mundial["away_avg_conceded"]

df_Mundial["strength_gap"] = abs(df_Mundial["diff_win_rate"])

In [ ]:
#Quitar partdios con hisotrial 0
df_model = df_Mundial[
    (df_Mundial["home_avg_goals"] > 0) |
    (df_Mundial["away_avg_goals"] > 0)
].copy()

In [ ]:
#Definimos X y Y/7 como los features que vamos ausar
y = df_model["resultado"]
features = [
    "home_avg_goals",
    "away_avg_goals",
    "home_avg_conceded",
    "away_avg_conceded",
    "home_win_rate",
    "away_win_rate",
    "diff_avg_goals",
    "diff_win_rate",
    "diff_avg_conceded",
    "strength_gap"
]

X = df_model[features]

In [ ]:
#Train y test definimos
train = df_model[df_model["Year"] <= 2014]
test  = df_model[df_model["Year"] >= 2018]

X_train = train[features]
y_train = train["resultado"]

X_test = test[features]
y_test = test["resultado"]

In [ ]:
#Verificamos tamaño
print(X_train.shape)
print(X_test.shape)

(825, 10)
(128, 10)


In [ ]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(
    n_estimators=300,
    max_depth=8,
    class_weight="balanced",
    random_state=42
)

model.fit(X_train, y_train)

RandomForestClassifier(class_weight='balanced', max_depth=8, n_estimators=300,
                       random_state=42)

In [ ]:
y_pred = model.predict(X_test)

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

[[27 11  7]
 [16  4  8]
 [13 14 28]]
              precision    recall  f1-score   support

          -1       0.48      0.60      0.53        45
           0       0.14      0.14      0.14        28
           1       0.65      0.51      0.57        55

    accuracy                           0.46       128
   macro avg       0.42      0.42      0.42       128
weighted avg       0.48      0.46      0.46       128



In [ ]:
#Split temporal
#df_Mundial['Match Date'].dt.year.unique()

array([1930, 1934, 1938, 1950, 1954, 1958, 1962, 1966, 1970, 1974, 1978,
       1982, 1986, 1990, 1994, 1998, 2002, 2006, 2010, 2014, 2018, 2022],
      dtype=int32)

In [ ]:
#Train y Test
#df_Mundial['Year'] = df_Mundial['Match Date'].dt.year
#train = df_Mundial[df_Mundial['Year'] <= 2014]
#test  = df_Mundial[df_Mundial['Year'] >= 2018]

In [ ]:
#Verificamos el volumenes de datos para entrenamiento y prueba
#print("Train:", train.shape)
#print("Test:", test.shape)

Train: (836, 39)
Test: (128, 39)


In [ ]:
#Verificamos si solamente esta utilizando 2018 y 2022
#test["Year"].unique()

array([2018, 2022], dtype=int32)

## Summary:

### Data Analysis Key Findings
*   The CSV file "FIFA World Cup 1930-2022 All Match Dataset.csv" was successfully loaded into a Pandas DataFrame named `df_Mundial`.
*   A `UnicodeDecodeError` encountered during previous attempts to load the file was resolved by specifying `encoding='latin1'` in the `pd.read_csv` function.
*   The first few rows of the DataFrame were displayed, confirming the correct ingestion of the data.

### Insights or Next Steps
*   The successful data loading with the correct encoding ensures that all characters are interpreted properly, preventing data corruption and enabling accurate subsequent analysis.
*   The loaded `df_Mundial` DataFrame is now prepared for further data cleaning, exploration, and analysis.
